In [1]:


import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# set or create an experiment
mlflow.set_experiment("exp8_sbert_lightgbm_smote_optuna_hpt") 


2025/12/02 11:09:53 INFO mlflow.tracking.fluent: Experiment with name 'exp8_sbert_lightgbm_smote_optuna_hpt' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/11', creation_time=1764654001057, experiment_id='11', last_update_time=1764654001057, lifecycle_stage='active', name='exp8_sbert_lightgbm_smote_optuna_hpt', tags={}>

In [7]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [8]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [ ]:

import os
import numpy as np
import joblib
import mlflow
import optuna
import lightgbm as lgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from imblearn.over_sampling import SMOTE

# -------------------------
# CONFIG
# -------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS = 3                        # inner CV for Optuna (keeps tuning time reasonable)
N_TRIALS = 25                     # adjust: 30-100 depending on time
EARLY_STOPPING_ROUNDS = 50
MLFLOW_EXPERIMENT_NAME = "SBERT_LightGBM_SMOTE_optuna_hpt"
SBERT_MODEL = "all-MiniLM-L6-v2"   # fast & high quality
TF_BATCH_SIZE = 256                # embedding batch size
N_JOBS = -1

# -------------------------
# Expect df to be in memory (same style as your Exp7)
# -------------------------
# Make sure df exists in the notebook/script (like your Exp7 workflow)
# df must contain 'text_clean' and 'sentiment_numeric' and numeric columns
# Example: df = pd.read_csv("data/processed/features.csv") if you prefer

# -------------------------
# Map target to integers consistently (same mapping as before)
# -------------------------
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['text_clean', 'sentiment_numeric']).reset_index(drop=True)
y = df['sentiment_numeric'].astype(int)

# -------------------------
# Numeric features (same approach as Exp7: all columns except first and last)
# -------------------------
# This mirrors your Exp7 slicing behaviour
if df.shape[1] > 2:
    X_numeric = df.iloc[:, 1:-1]
    scaler = StandardScaler(with_mean=False)
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    has_numeric = True
else:
    X_numeric_scaled = None
    has_numeric = False

# -------------------------
# Create SBERT embeddings (same as Exp7)
# -------------------------
print("Loading SBERT model:", SBERT_MODEL)
sbert = SentenceTransformer(SBERT_MODEL)

texts = df['text_clean'].astype(str).tolist()
embeddings = []
print("Creating SBERT embeddings (batches):")
for i in tqdm(range(0, len(texts), TF_BATCH_SIZE)):
    batch = texts[i:i+TF_BATCH_SIZE]
    emb = sbert.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)   # shape: (n_samples, emb_dim)
print("Embeddings shape:", embeddings.shape)

# Combine text embeddings + numeric features (if any) — same format as Exp7
if has_numeric:
    from scipy import sparse
    X_numeric_arr = np.asarray(X_numeric_scaled)
    X = np.hstack([embeddings, X_numeric_arr])
else:
    X = embeddings

# -------------------------
# Train/test split (stratified) — same as Exp7
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# -------------------------
# Apply SMOTE on training set to reduce class imbalance 
# -------------------------
print("Train distribution before SMOTE:", dict(zip(*np.unique(y_train_full, return_counts=True))))
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_bal, y_train_bal = sm.fit_resample(X_train_full, y_train_full)
print("Train distribution after SMOTE:", dict(zip(*np.unique(y_train_bal, return_counts=True))))

# -------------------------
# compute sample/class weights on (balanced) training set 
# -------------------------
classes = np.unique(y_train_bal)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_bal)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
sample_weight_full = np.array([class_weight_dict[int(lbl)] for lbl in y_train_bal])

print("Class weights (from balanced train):", class_weight_dict)

# -------------------------
# Optuna objective (uses StratifiedKFold CV) — maximize macro recall
# -------------------------
def objective(trial):
    params = {
        "boosting_type": "gbdt",
        "objective": "multiclass",
        "num_class": len(classes),
        "metric": "multi_logloss",
        "n_jobs": N_JOBS,
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
        # keep class_weight as dictionary so LightGBM can use it (as in Exp7)
        "class_weight": class_weight_dict,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "random_state": RANDOM_STATE
    }

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    recalls = []

    # use stratified folds and early stopping on fold validation
    for train_idx, val_idx in skf.split(X_train_bal, y_train_bal):
        X_tr, X_val = X_train_bal[train_idx], X_train_bal[val_idx]
        y_tr, y_val = pd.Series(y_train_bal[train_idx]), pd.Series(y_train_bal[val_idx])

        sw_tr = np.array([class_weight_dict[int(lbl)] for lbl in y_tr])

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
        )

        preds = model.predict(X_val)
        r = recall_score(y_val, preds, average="macro")
        recalls.append(r)

    # return mean macro recall across folds
    return float(np.mean(recalls))

# run study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best optuna params:", study.best_params)


# -------------------------
best_params = study.best_params.copy()
best_params.update({
    "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(classes),
    "metric": "multi_logloss",
    "class_weight": class_weight_dict,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "verbosity": -1
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train_bal, y_train_bal,
    sample_weight=sample_weight_full,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)]
)

# -------------------------
# Evaluate (with confidence thresholding) — keep printing style like Exp7
# -------------------------
probs = final_model.predict_proba(X_test)
y_pred = np.argmax(probs, axis=1)

# Force low-confidence predictions to Neutral (class 0)
CONF_THRESH = 0.45
low_conf_mask = probs.max(axis=1) < CONF_THRESH
y_pred[low_conf_mask] = 0

acc = accuracy_score(y_test, y_pred)
macro_rec = recall_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nFINAL RESULTS (Exp8)")
print("Accuracy:", acc)
print("Macro Recall:", macro_rec)
print(report)
print("Confusion Matrix:\n", cm)

# -------------------------
# MLflow logging & saving artifacts (Exp7-like)
# -------------------------
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name="Exp8_SBERT_LGBM_SMOTE"):
    mlflow.log_params(best_params)
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("macro_recall", float(macro_rec))

    # save confusion matrix image
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix -  SBERT+LGBM (Exp8)")
    plt.savefig("exp8_confusion_matrix.png")
    mlflow.log_artifact("exp8_confusion_matrix.png")
    plt.close()

    # save model + sbert + scaler
    joblib.dump(final_model, "exp8_lgbm_model.pkl")
    joblib.dump(sbert, "exp8_sbert_model.pkl")        # sentence-transformer object
    if has_numeric:
        joblib.dump(scaler, "exp8_numeric_scaler.pkl")
    mlflow.log_artifact("exp8_lgbm_model.pkl")
    mlflow.log_artifact("exp8_sbert_model.pkl")
    if has_numeric:
        mlflow.log_artifact("exp8_numeric_scaler.pkl")

print("Artifacts saved. Done.")


Loading SBERT model: all-MiniLM-L6-v2
Creating SBERT embeddings (batches):


  0%|          | 0/194 [00:00<?, ?it/s]

KeyboardInterrupt: 